# 🔗 LangChain — Complete End-to-End Guide (100% FREE)

**IIST Agentic AI Training Program · Days 8–9**

---

### Zero Cost Stack

| Component | Free Solution | Why |
|-----------|---------------|-----|
| **LLM** | OpenRouter free models (Llama 3.3 70B, Gemma 3, etc.) | OpenAI-compatible API, no credit card |
| **Embeddings** | HuggingFace `all-MiniLM-L6-v2` (runs locally) | Open-source, runs on Colab CPU |
| **Vector DB** | ChromaDB (in-memory) | Open-source, no setup |
| **Agents** | LangGraph `create_react_agent` | Open-source |

### What We'll Cover

| # | Topic | What You'll Build |
|---|-------|-------------------|
| 1 | **Setup & First Call** | OpenRouter free model via LangChain |
| 2 | **Prompt Templates** | Reusable, parameterized prompts |
| 3 | **LCEL (Pipe Operator)** | Composable chains: prompt \| model \| parser |
| 4 | **Output Parsers** | Structured JSON/Pydantic outputs |
| 5 | **Memory** | Conversational chains that remember context |
| 6 | **Tools** | Custom tools + tool calling |
| 7 | **Agents (ReAct)** | LLM that autonomously picks and uses tools |
| 8 | **RAG Chain** | Retrieval-Augmented Generation with ChromaDB + HuggingFace embeddings |
| 9 | **Routing & Parallel** | Conditional chains + parallel execution |
| 10 | **Streaming** | Real-time token-by-token output |

---

> **Prerequisites**: Basic Python. No prior LangChain experience needed.
>
> **Free API Key Required**: Sign up at [openrouter.ai](https://openrouter.ai) — no credit card needed.

---
## 1. Setup & Installation

In [ ]:
# Install all packages
!pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-core \
    langchain-huggingface \
    langgraph \
    chromadb \
    pydantic \
    sentence-transformers

In [ ]:
import os

# ══════════════════════════════════════════════
# SET YOUR FREE OPENROUTER API KEY
# ══════════════════════════════════════════════
# 1. Go to https://openrouter.ai and sign up (no credit card needed)
# 2. Go to https://openrouter.ai/keys and create a key
# 3. Paste it below

OPENROUTER_API_KEY = "sk-or-..."  # <-- Replace with your key

# --- OR use Colab Secrets (recommended for sharing) ---
# from google.colab import userdata
# OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY

In [ ]:
# ══════════════════════════════════════════════
# CONFIGURE THE FREE LLM
# ══════════════════════════════════════════════
# OpenRouter is OpenAI-API-compatible, so we use ChatOpenAI
# with a different base_url. That's the only trick!

from langchain_openai import ChatOpenAI

# ── Free models available on OpenRouter (pick one) ──
# "meta-llama/llama-3.3-70b-instruct:free"   -- Best free all-rounder
# "google/gemma-3-27b-it:free"                -- Strong, Google's open model
# "mistralai/mistral-small-3.1-24b-instruct:free" -- Fast, good for chains
# "nvidia/llama-3.1-nemotron-70b-instruct:free"   -- Great reasoning
# "qwen/qwen3-235b-a22b:free"                -- Massive, high quality
# "deepseek/deepseek-r1:free"                 -- Strong reasoning

FREE_MODEL = "meta-llama/llama-3.3-70b-instruct:free"

llm = ChatOpenAI(
    model=FREE_MODEL,
    temperature=0,
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

# Quick test
response = llm.invoke("Say 'Hello, I am a free LLM running via OpenRouter!' and nothing else.")
print(f"Model: {FREE_MODEL}")
print(f"Response: {response.content}")
print("\n✅ Free LLM is working!")

---
## 2. Side-by-Side: Raw API vs LangChain

Why does LangChain exist? Let's see the same task done both ways.

In [ ]:
# ═══════════════════════════════════════
# METHOD 1: Raw OpenAI SDK (pointing to OpenRouter)
# ═══════════════════════════════════════
from openai import OpenAI

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

response = client.chat.completions.create(
    model=FREE_MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is LangChain in one sentence?"}
    ]
)
print("[Raw OpenAI SDK → OpenRouter]")
print(response.choices[0].message.content)

In [ ]:
# ═══════════════════════════════════════
# METHOD 2: LangChain (same model, composable)
# ═══════════════════════════════════════
from langchain_core.messages import SystemMessage, HumanMessage

response = llm.invoke([
    SystemMessage(content="You are a helpful assistant."),
    HumanMessage(content="What is LangChain in one sentence?")
])
print("[LangChain → OpenRouter]")
print(response.content)

### 💡 Key Takeaway

Same result! But LangChain gives you composability (LCEL), built-in memory, agents, RAG — and you can swap the model by changing ONE line. Today we're using Llama 3.3 for free; tomorrow you could switch to GPT-4o or Claude by just changing `model=` and `base_url=`.

---
## 3. Prompt Templates

Reusable, parameterized prompts — the building blocks of every chain.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# ── Simple template ──
simple_prompt = ChatPromptTemplate.from_template(
    "Explain {topic} in simple terms for a 10-year-old. Keep it to 2-3 sentences."
)

# See what it produces
messages = simple_prompt.format_messages(topic="quantum computing")
print("Formatted prompt:")
for msg in messages:
    print(f"  [{msg.type}]: {msg.content}")

In [ ]:
# ── Multi-role template (system + user) ──
expert_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert {role}. Keep answers practical and under 100 words."),
    ("human", "{question}")
])

response = llm.invoke(
    expert_prompt.format_messages(
        role="Python developer",
        question="What's the best way to handle errors in async Python?"
    )
)
print(response.content)

---
## 4. LCEL — The Pipe Operator (This Changes Everything)

**LCEL** = LangChain Expression Language. Compose components with `|` like Unix pipes:

```
prompt | model | parser
```

Data flows left → right. Each component transforms and passes.

In [ ]:
from langchain_core.output_parsers import StrOutputParser

# ═══════════════════════════════════════
# YOUR FIRST LCEL CHAIN
# ═══════════════════════════════════════
#
# Data flow:
#   {"topic": "..."}
#       ↓
#   prompt template (fills variables)
#       ↓
#   LLM (generates response)
#       ↓
#   StrOutputParser (extracts clean text)
#       ↓
#   "clean string output"

prompt = ChatPromptTemplate.from_template(
    "Give me 3 fun facts about {topic}. Keep each fact to one sentence."
)

# Compose with the pipe operator
chain = prompt | llm | StrOutputParser()

# Invoke!
result = chain.invoke({"topic": "the Moon"})
print(result)
print(f"\nReturn type: {type(result)}")  # Clean string, not AIMessage

In [ ]:
# ═══════════════════════════════════════
# MULTI-STEP CHAIN (Research Pipeline)
# ═══════════════════════════════════════
# Step 1: Summarize a topic
# Step 2: Generate quiz questions from the summary

# Step 1 chain
summarize_prompt = ChatPromptTemplate.from_template(
    "Summarize '{topic}' in 3 bullet points. Be concise (under 50 words total)."
)
summarize_chain = summarize_prompt | llm | StrOutputParser()

# Step 2 chain
quiz_prompt = ChatPromptTemplate.from_template(
    "Based on this summary, create 2 multiple-choice quiz questions:\n\n{summary}"
)
quiz_chain = quiz_prompt | llm | StrOutputParser()

# Compose: summary output feeds into quiz input
full_pipeline = (
    summarize_chain
    | (lambda summary: {"summary": summary})
    | quiz_chain
)

result = full_pipeline.invoke({"topic": "photosynthesis"})
print(result)

### 💡 Why LCEL Matters

Every LCEL chain automatically supports:
- `.invoke()` — run once
- `.stream()` — stream tokens
- `.batch()` — run on multiple inputs
- `.ainvoke()` — async

Build once, get streaming/batching/async for free.

---
## 5. Output Parsers — Structured Data from LLMs

LLMs return text. Output parsers extract **structured objects**.

In [ ]:
# ═══════════════════════════════════════
# PYDANTIC OUTPUT PARSER
# ═══════════════════════════════════════
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List

# Define the shape of data you want
class MovieReview(BaseModel):
    title: str = Field(description="The movie title")
    rating: float = Field(description="Rating out of 10")
    pros: List[str] = Field(description="List of positive aspects")
    cons: List[str] = Field(description="List of negative aspects")
    verdict: str = Field(description="One-line verdict")

parser = PydanticOutputParser(pydantic_object=MovieReview)

prompt = ChatPromptTemplate.from_template(
    "Review the movie '{movie}'.\n\n{format_instructions}"
)

chain = prompt | llm | parser

review = chain.invoke({
    "movie": "Inception",
    "format_instructions": parser.get_format_instructions()
})

print(f"Title: {review.title}")
print(f"Rating: {review.rating}/10")
print(f"Pros: {review.pros}")
print(f"Cons: {review.cons}")
print(f"Verdict: {review.verdict}")
print(f"\nType: {type(review)}")  # It's a Pydantic object!

In [ ]:
# ═══════════════════════════════════════
# with_structured_output (cleaner API)
# ═══════════════════════════════════════
# Note: This uses tool calling under the hood.
# Works with Llama 3.3 on OpenRouter since it supports tool calling.

class CityInfo(BaseModel):
    """Information about a city."""
    name: str = Field(description="City name")
    country: str = Field(description="Country")
    population_millions: float = Field(description="Approximate population in millions")
    famous_for: List[str] = Field(description="What the city is famous for")

structured_llm = llm.with_structured_output(CityInfo)

city = structured_llm.invoke("Tell me about Tokyo")

print(f"City: {city.name}, {city.country}")
print(f"Population: ~{city.population_millions}M")
print(f"Famous for: {', '.join(city.famous_for)}")

---
## 6. Memory — Conversational Chains

LLMs are stateless. Memory lets them remember conversation history.

In [ ]:
from langchain_core.messages import AIMessage
from langchain_core.prompts import MessagesPlaceholder

# Prompt with a history slot
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a friendly tutor. Keep answers brief (2-3 sentences max)."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = chat_prompt | llm | StrOutputParser()

# ── Multi-turn conversation ──
history = []

def chat(user_message):
    response = chain.invoke({"input": user_message, "history": history})
    history.append(HumanMessage(content=user_message))
    history.append(AIMessage(content=response))
    return response

# Turn 1
print("You: My name is Soham and I'm learning about AI agents.")
print(f"AI: {chat('My name is Soham and I am learning about AI agents.')}\n")

# Turn 2 — does it remember?
print("You: What's my name and what am I learning?")
print(f"AI: {chat('Whats my name and what am I learning?')}\n")

# Turn 3
print("You: Give me a simple definition of an agent.")
print(f"AI: {chat('Give me a simple definition of an agent.')}")

print(f"\n📊 History: {len(history)} messages")

---
## 7. Tools — Giving LLMs Superpowers

Tools let the LLM interact with the real world. This is the bridge from *chatbot* to *agent*.

In [ ]:
from langchain_core.tools import tool

# ── Define custom tools using the @tool decorator ──
# The docstring is CRITICAL — the LLM reads it to decide when to use the tool

@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression. Use Python syntax.
    Examples: '2 + 3', '45 * 12', '2 ** 10', 'round(3.14159, 2)'"""
    try:
        import math
        result = eval(expression, {"__builtins__": {}, "math": math,
                                    "round": round, "abs": abs, "pow": pow})
        return f"Result: {result}"
    except Exception as e:
        return f"Error: {e}"

@tool
def get_word_count(text: str) -> str:
    """Count the number of words in a given text."""
    return f"The text contains {len(text.split())} words."

@tool
def lookup_capital(country: str) -> str:
    """Look up the capital city of a country."""
    capitals = {
        "india": "New Delhi", "france": "Paris", "japan": "Tokyo",
        "brazil": "Brasília", "australia": "Canberra", "germany": "Berlin",
        "usa": "Washington, D.C.", "united states": "Washington, D.C.",
    }
    result = capitals.get(country.lower())
    return f"The capital of {country} is {result}." if result else f"Capital not found for {country}."

# Inspect a tool
print(f"Name: {calculate.name}")
print(f"Description: {calculate.description}")
print(f"Schema: {calculate.args_schema.model_json_schema()}")

In [ ]:
# ═══════════════════════════════════════
# BIND TOOLS TO THE MODEL
# ═══════════════════════════════════════
tools = [calculate, get_word_count, lookup_capital]
llm_with_tools = llm.bind_tools(tools)

# The LLM now CHOOSES to call a tool
response = llm_with_tools.invoke("What is 247 * 389?")

print("Content:", response.content)
print("\nTool calls:")
for tc in response.tool_calls:
    print(f"  Tool: {tc['name']}, Args: {tc['args']}")

In [ ]:
# ═══════════════════════════════════════
# MANUAL TOOL EXECUTION (understand the loop!)
# ═══════════════════════════════════════
# This is what happens INSIDE an agent:
# 1. LLM decides to call a tool
# 2. WE execute the tool
# 3. We feed the result back to the LLM
# 4. LLM gives the final answer

from langchain_core.messages import ToolMessage

# Step 1: LLM chooses a tool
ai_message = llm_with_tools.invoke("What's the capital of India?")
print("Step 1 — LLM wants to call:", ai_message.tool_calls)

# Step 2: Execute the tool ourselves
tool_call = ai_message.tool_calls[0]
tool_map = {t.name: t for t in tools}
tool_result = tool_map[tool_call["name"]].invoke(tool_call["args"])
print(f"Step 2 — Tool result: {tool_result}")

# Step 3: Send result back to get final answer
messages = [
    HumanMessage(content="What's the capital of India?"),
    ai_message,
    ToolMessage(content=tool_result, tool_call_id=tool_call["id"])
]
final = llm_with_tools.invoke(messages)
print(f"Step 3 — Final answer: {final.content}")

### 💡 This Manual Loop IS the Agent

The 3 steps above are exactly what an **agent** automates:
1. LLM decides → tool call
2. Execute tool → get result  
3. Feed result back → LLM decides again (repeat or answer)

Next section puts this loop on autopilot.

---
## 8. Agents — The ReAct Loop on Autopilot

An agent automates the tool-calling loop. LangGraph's `create_react_agent` handles everything.

In [ ]:
from langgraph.prebuilt import create_react_agent

# One line creates a full ReAct agent!
agent = create_react_agent(llm, tools)

# Helper to display agent traces nicely
def ask_agent(question, agent_instance=agent):
    print(f"\n{'═' * 60}")
    print(f"🧑 {question}")
    print(f"{'═' * 60}")
    result = agent_instance.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        if msg.type == 'ai' and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  🔧 Calls: {tc['name']}({tc['args']})")
        elif msg.type == 'tool':
            print(f"  📋 Result: {msg.content[:150]}")
        elif msg.type == 'ai' and msg.content:
            print(f"\n🤖 {msg.content}")

# Test!
ask_agent("What is 2^10 + 3^5?")

In [ ]:
# Multi-tool query — agent decides which tools to use!
ask_agent("What's the capital of Japan? Also calculate 100 / 3 rounded to 2 decimals.")

In [ ]:
# No tools needed — agent answers directly
ask_agent("What is the ReAct pattern in AI?")

---
## 9. RAG — Retrieval-Augmented Generation (100% Free)

RAG lets the LLM answer questions about **your own data**.

Our free stack:
- **Embeddings**: HuggingFace `all-MiniLM-L6-v2` (runs locally, no API needed)
- **Vector DB**: ChromaDB (in-memory)
- **LLM**: OpenRouter free model

In [ ]:
# ═══════════════════════════════════════
# STEP 1: Set up FREE embeddings (HuggingFace)
# ═══════════════════════════════════════
# This downloads a small model (~80MB) and runs it locally on Colab.
# No API key, no cost, no rate limits.

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
)

# Quick test
test_embedding = embeddings.embed_query("Hello world")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")
print("\n✅ Free HuggingFace embeddings working!")

In [ ]:
# ═══════════════════════════════════════
# STEP 2: Create documents and store in ChromaDB
# ═══════════════════════════════════════
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Sample knowledge base
documents = [
    Document(
        page_content="LangChain is a framework for building applications powered by LLMs. "
        "It provides tools for prompt management, chaining LLM calls, and integrating with external data sources. "
        "Created by Harrison Chase, it was first released in October 2022.",
        metadata={"source": "langchain_overview", "topic": "framework"}
    ),
    Document(
        page_content="LCEL (LangChain Expression Language) is the modern way to build chains in LangChain. "
        "It uses the pipe operator (|) to compose prompts, models, and parsers into declarative pipelines. "
        "LCEL chains automatically support streaming, batching, and async execution.",
        metadata={"source": "lcel_guide", "topic": "lcel"}
    ),
    Document(
        page_content="Agents in LangChain use the ReAct pattern: the LLM reasons about what to do, takes an action "
        "(like calling a tool), observes the result, and repeats until the task is complete. "
        "Agents are different from chains because they make dynamic decisions at runtime.",
        metadata={"source": "agents_guide", "topic": "agents"}
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) grounds LLM responses in external knowledge. "
        "The pipeline involves chunking documents, creating embeddings, storing in a vector database, "
        "and retrieving relevant chunks at query time to provide context to the LLM.",
        metadata={"source": "rag_guide", "topic": "rag"}
    ),
    Document(
        page_content="LangGraph extends LangChain with state machines, persistent checkpoints, and human-in-the-loop. "
        "It is recommended for production-grade agents that need reliability and control flow. "
        "LangGraph models agents as graphs with nodes (actions) and edges (transitions).",
        metadata={"source": "langgraph_intro", "topic": "langgraph"}
    ),
    Document(
        page_content="ChromaDB, Pinecone, and Qdrant are popular vector databases for storing embeddings. "
        "ChromaDB is open-source and runs locally, ideal for development. "
        "Pinecone and Qdrant offer managed cloud solutions for production workloads.",
        metadata={"source": "vectordb_comparison", "topic": "vector_db"}
    ),
]

# Split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
chunks = splitter.split_documents(documents)
print(f"Split {len(documents)} docs → {len(chunks)} chunks")

# Store in ChromaDB with HuggingFace embeddings (FREE!)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,  # HuggingFace, not OpenAI!
    collection_name="langchain_guide"
)
print(f"Stored {vectorstore._collection.count()} vectors in ChromaDB")
print("\n✅ All free — no API costs for embeddings!")

In [ ]:
# ═══════════════════════════════════════
# STEP 3: Build the RAG Chain
# ═══════════════════════════════════════
from langchain_core.runnables import RunnablePassthrough

# Create retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# RAG prompt
rag_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say "I don't have enough information."

Context:
{context}

Question: {question}

Answer (be concise):""")

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# The RAG chain
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

# Test it!
questions = [
    "What is LCEL and why does it matter?",
    "What's the difference between agents and chains?",
    "Which vector database should I use for development?",
]

for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_chain.invoke(q)}")
    print()

---
## 10. Routing & Parallel Execution

In [ ]:
# ═══════════════════════════════════════
# PARALLEL EXECUTION — 3 chains run simultaneously
# ═══════════════════════════════════════
from langchain_core.runnables import RunnableParallel

summary_chain = (
    ChatPromptTemplate.from_template("Summarize {topic} in 1 sentence.")
    | llm | StrOutputParser()
)
pros_chain = (
    ChatPromptTemplate.from_template("List 2 advantages of {topic}. Brief bullet points.")
    | llm | StrOutputParser()
)
cons_chain = (
    ChatPromptTemplate.from_template("List 2 challenges of {topic}. Brief bullet points.")
    | llm | StrOutputParser()
)

parallel_chain = RunnableParallel(
    summary=summary_chain,
    pros=pros_chain,
    cons=cons_chain
)

result = parallel_chain.invoke({"topic": "using AI agents in production"})

print("📝 Summary:")
print(result["summary"])
print("\n✅ Pros:")
print(result["pros"])
print("\n⚠️ Challenges:")
print(result["cons"])

In [ ]:
# ═══════════════════════════════════════
# CONDITIONAL ROUTING
# ═══════════════════════════════════════
from langchain_core.runnables import RunnableBranch

math_chain = (
    ChatPromptTemplate.from_template("You are a math tutor. Solve step by step (keep brief): {input}")
    | llm | StrOutputParser()
)
code_chain = (
    ChatPromptTemplate.from_template("You are a Python expert. Answer with a short code example: {input}")
    | llm | StrOutputParser()
)
general_chain = (
    ChatPromptTemplate.from_template("Answer helpfully and briefly: {input}")
    | llm | StrOutputParser()
)

router = RunnableBranch(
    (lambda x: any(w in x["input"].lower() for w in ["calculate", "math", "solve"]), math_chain),
    (lambda x: any(w in x["input"].lower() for w in ["code", "python", "function"]), code_chain),
    general_chain
)

print("--- Math Route ---")
print(router.invoke({"input": "Solve: what is the derivative of x^3 + 2x?"}))
print("\n--- Code Route ---")
print(router.invoke({"input": "Write a Python function to check if a number is prime"}))
print("\n--- General Route ---")
print(router.invoke({"input": "What is the capital of France?"}))

---
## 11. Streaming — Real-Time Output

In [ ]:
# Every LCEL chain supports .stream() for free!
stream_chain = (
    ChatPromptTemplate.from_template("Write a short 4-line poem about {topic}.")
    | llm
    | StrOutputParser()
)

print("Streaming (token by token):")
print("-" * 40)
for chunk in stream_chain.stream({"topic": "coding at midnight"}):
    print(chunk, end="", flush=True)
print("\n" + "-" * 40 + "\nDone!")

---
## 12. Full System — Agent + RAG + Tools (All Free)

Combining everything into one agent that can search our knowledge base, do math, and answer questions.

In [ ]:
@tool
def search_knowledge_base(query: str) -> str:
    """Search the LangChain knowledge base for information about
    LangChain, LCEL, agents, RAG, LangGraph, or vector databases.
    Use this when the user asks about any of these topics."""
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information found."
    return "\n\n".join(
        f"[{doc.metadata.get('source', '?')}]: {doc.page_content}"
        for doc in docs
    )

@tool
def estimate_llm_cost(prompt_tokens: int, completion_tokens: int, model: str = "gpt-4o-mini") -> str:
    """Estimate the cost of an LLM API call given token counts.
    Supports models: gpt-4o-mini, gpt-4o, claude-3-sonnet"""
    pricing = {
        "gpt-4o-mini": {"input": 0.15, "output": 0.60},
        "gpt-4o": {"input": 2.50, "output": 10.00},
        "claude-3-sonnet": {"input": 3.00, "output": 15.00},
    }
    if model not in pricing:
        return f"Unknown model. Supported: {list(pricing.keys())}"
    p = pricing[model]
    total = (prompt_tokens / 1e6) * p["input"] + (completion_tokens / 1e6) * p["output"]
    return f"Estimated cost for {model}: ${total:.6f}"

# Create the full agent
full_agent = create_react_agent(llm, [search_knowledge_base, estimate_llm_cost, calculate])

# Test!
ask_agent("What is LCEL and why should I use it?", full_agent)
ask_agent("How much would 50000 input tokens + 10000 output tokens cost with gpt-4o?", full_agent)
ask_agent("What's the difference between LangChain and LangGraph?", full_agent)

---
## 🎯 Summary & Cost Report

### Everything We Built — For Free

```
┌─────────────────────────────────────────────────────────┐
│                   FREE STACK USED                        │
├──────────────────┬──────────────────────────────────────┤
│ LLM              │ Llama 3.3 70B via OpenRouter (free)  │
│ Embeddings       │ HuggingFace all-MiniLM-L6-v2 (local)│
│ Vector DB        │ ChromaDB (in-memory, open-source)    │
│ Agent Framework  │ LangGraph (open-source)              │
│ Total Cost       │ $0.00                                │
└──────────────────┴──────────────────────────────────────┘
```

### Key Concepts

| Concept | One-liner |
|---------|----------|
| **LCEL** | Compose chains with `\|` — get streaming/batch/async for free |
| **Prompt Templates** | Reusable prompts with variables and roles |
| **Output Parsers** | Get Pydantic objects from LLM text |
| **Tools** | Python functions the LLM can choose to call |
| **Agents** | Automated think → act → observe → repeat loop |
| **RAG** | Ground LLM in your data via embeddings + retrieval |
| **Memory** | Maintain conversation context across turns |
| **Routing** | Send queries to specialized chains |

### Swapping Models

The beauty of this setup — changing the model is ONE line:

```python
# Free models on OpenRouter (no credit card)
FREE_MODEL = "meta-llama/llama-3.3-70b-instruct:free"
FREE_MODEL = "google/gemma-3-27b-it:free"
FREE_MODEL = "qwen/qwen3-235b-a22b:free"
FREE_MODEL = "deepseek/deepseek-r1:free"

# Paid models (when you're ready to scale)
# Just change base_url back to OpenAI/Anthropic
```

### What's Next

- **LangGraph** — Stateful agents with checkpoints (Days 10-11)
- **Advanced RAG** — HyDE, rerankers, multi-query (Days 6-7)
- **n8n** — Visual workflow automation (Days 12-13)
- **Multi-Agent** — CrewAI, AutoGen (Day 14)

---

*IIST Agentic AI Training Program · Module 4 · Days 8-9*

*Build agents that think. Deploy systems that scale. Ship AI that works.*